<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/28_my_voice_backgrounds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 28 — Kendi sesim + veri setinden arka planlar → modellerin skoru (EXP-033; keşifsel, GPU)

**Ne yapıyor?** `data_derived/alper_mix_v1/my_voices/` içindeki 7 kaydın her birine, `background/` klasöründeki 13 arka planı **kodla** ekler (seviye korunmuş; bağışçının konuşma / arka plan oranı aynı) → 7 × 14 = 98 ses. Sonra her sesi modellere verir:
- **MFCC-44 + LR**
- **Dondurulmuş gömme + LR:** CNN10, CNN14, BEATs, WavLM Base+, WavLM Large
- **Fine-tune:** CNN10, CNN14, BEATs (yeniden eğitilir; Faz 3 ağırlıkları saklanmamıştı)

**Ezberleme önlemi:** bütün modeller, arka planı kullanılan 13 kişi **ve onların kayıt günlerindeki herkes** çıkarılarak yeniden eğitilir; 14 sürüm aynı modele girer.

**Nasıl okunur?** Senin sesin veri setinin dışında; orijinal kaydın skoru keyfi olabilir. Bakılacak şey **aynı kaydın orijinaline göre kayması** (logit farkı) ve bu kaymanın arka planın **astımlı / sağlıklı** bağışçıdan gelmesiyle ilişkisi. İstatistik yok: gözle görmek ve şüphe üretmek için.

*Runtime → T4 GPU.* Süre: karıştırma ~2 dk, MFCC ~1 dk, dondurulmuş ~10–15 dk, fine-tune ~30–40 dk (isteğe bağlı) [tahmin].

## 0. Oturum kurulumu

In [ ]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook T4 GPU ister: Runtime → Change runtime type → T4 GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
SPLITS_DAY  = DERIVED / 'splits_day'
MIX_IN      = DERIVED / 'alper_mix_v1'                    # Alper: background/ (L03b.wav …) ve my_voices/ (aaaaaa.m4a …)
BG_DRIVE    = EXPERIMENTS / 'background_v1'               # EXP-032: seviye korunmuş izler + dinleme anahtarı
FT_OUT      = EXPERIMENTS / 'phase3' / 'finetune'         # Faz 3 kalibrasyon eğrileri (epoch sayısı buradan)
OUT         = Path('/content/my_voice_v1')                # önce yerel, sonra Drive'a kopya
OUT_DRIVE   = EXPERIMENTS / 'my_voice_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
for p in [MIX_IN / 'background', MIX_IN / 'my_voices', BG_DRIVE / 'tracks_32k.f32', BG_DRIVE / 'tracks_index.csv',
          BG_DRIVE / 'listening' / 'listening_key.csv', DERIVED / 'participant_context.csv', SPLITS_DAY / 'outer_r0.csv',
          DERIVED / 'mfcc' / 'features_cache.csv', EMB_DIR]:
    assert p.exists(), f'Bulunamadı: {p}'
print('arka planlar:', sorted(p.name for p in (MIX_IN / 'background').glob('*.wav')))
print('sesler:', sorted(p.name for p in (MIX_IN / 'my_voices').iterdir()))


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

In [ ]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

In [ ]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
for p in ['scripts/my_voice_demo.py', 'tests/test_my_voice.py', 'scripts/background_extract.py']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: PR birleştirildi mi?')
sh(f'cd {REPO_DIR} && python tests/test_my_voice.py')
sys.path.insert(0, str(REPO_DIR / 'scripts'))
import my_voice_demo as MV
import pandas as pd, numpy as np
COMMON = f'--context {DERIVED}/participant_context.csv --splits-dir {SPLITS_DAY} --out-dir {OUT}'

## 1. Karıştırma (98 ses)

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/my_voice_demo.py mix --voice-dir "{MIX_IN}/my_voices" --bg-dir "{MIX_IN}/background" '
   f'--key {BG_DRIVE}/listening/listening_key.csv --tracks-dir {BG_DRIVE} --context {DERIVED}/participant_context.csv --out-dir {OUT}')
idx = pd.read_csv(OUT / 'mixcache' / 'index.csv')
assert len(idx) == idx.slot.nunique() * (1 + idx.bg.nunique() - 1), 'beklenen 7 × 14'
print(len(idx), 'ses;', idx.slot.nunique(), 'öğe ×', idx.bg.nunique(), 'sürüm')
shutil.copytree(OUT, OUT_DRIVE, dirs_exist_ok=True); print('Drive\'a kopyalandı:', OUT_DRIVE)

## 2. MFCC-44 + LR

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/my_voice_demo.py score-mfcc {COMMON} --features {DERIVED}/mfcc/features_cache.csv')
shutil.copytree(OUT, OUT_DRIVE, dirs_exist_ok=True); print('Drive\'a kopyalandı:', OUT_DRIVE)

## 3. Dondurulmuş gömmeler + LR
Karışımların gömmeleri veri setiyle aynı kodla (`extract_embeddings.py`, 4 s / 2 s pencere) çıkarılır.

In [ ]:
FROZEN = ['cnn10', 'cnn14', 'beats', 'wavlm_base_plus', 'wavlm_large']
for bb in FROZEN:
    sh(f'cd {REPO_DIR} && python scripts/extract_embeddings.py --backbone {bb} --model-dir {MODELS} --cache-dir {OUT}/mixcache '
       f'--out-dir {OUT}/mix_emb/{bb} --views win4')
    sh(f'cd {REPO_DIR} && python scripts/my_voice_demo.py score-frozen {COMMON} --backbone {bb} --emb-root {EMB_DIR} '
       f'--mix-emb {OUT}/mix_emb/{bb}')
shutil.copytree(OUT, OUT_DRIVE, dirs_exist_ok=True); print('Drive\'a kopyalandı:', OUT_DRIVE)

## 4. Fine-tune (isteğe bağlı, ~30–40 dk)
Faz 3'ün ağırlıkları saklanmadığı için her kol bir kez, dışlama sonrası eğitim kümesinde, Faz 3 kalibrasyonunun seçtiği epoch sayısıyla (D-043) eğitilir. Ana önbellek (~4.6 GB) önce yerel diske kopyalanır. Atlamak için `RUN_FT = False`.

In [ ]:
RUN_FT = True
if RUN_FT:
    info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
    LOCAL_CACHE.mkdir(exist_ok=True)
    for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
        dst = LOCAL_CACHE / name
        if not dst.exists() or sha256(dst) != info['files_sha256'][name]:
            tmp = dst.with_name(name + '.part'); shutil.copy(DRIVE_CACHE / name, tmp); tmp.rename(dst)
        assert sha256(dst) == info['files_sha256'][name], f'{name}: sha256 uyuşmuyor'
    for arm in ['cnn10', 'cnn14', 'beats']:
        sh(f'cd {REPO_DIR} && python scripts/my_voice_demo.py score-ft {COMMON} --arm {arm} --cache-dir {LOCAL_CACHE} '
           f'--ft-root {FT_OUT} --model-dir {MODELS} --work-dir /content/my_voice_ft')

    shutil.copytree(OUT, OUT_DRIVE, dirs_exist_ok=True); print('Drive\'a kopyalandı:', OUT_DRIVE)

## 5. Arayüz
- **Ses:** öğe seç (ya da "hepsi": 7 öğenin ortalaması).
- **Modeller:** bir ya da birkaç model.
- **Gösterim:** *kayma* (orijinale göre logit farkı; + = astıma doğru) ya da *astım olasılığı*.
- Satırlar: orijinal, sonra astımlı bağışçıların arka planları (kırmızı), sonra sağlıklılarınki (mavi). Parantezde tür (A1 oda tonu / A2 tüm sessizlik / B durağan sentez) ve dönem (erken = son sağlıklı kayda kadar; geç = sonrası, yalnız astımlı).
- **Dinle:** seçili öğenin istediğin sürümünü çalar.

Not: B türü 0b'de konuşma spektrumu taşıdı (cinsiyet AUC 0.93); B'deki kaymayı "ortam" diye okuma.

In [ ]:
import ipywidgets as W, matplotlib.pyplot as plt
from IPython.display import display, Audio, clear_output
S = MV.load_all(OUT)
tur = {'A1': 'oda tonu', 'A2': 'tüm sessizlik', 'B': 'durağan sentez'}
S['arka_plan'] = np.where(S.bg == 'orig', 'orijinal',
                          S.bg + ' · ' + S.bg_group + ' · ' + S.bg_variant.map(tur).fillna('') + ' · ' + S.bg_period)
fam_order = {'MFCC': 0, 'dondurulmuş': 1, 'fine-tune': 2}
MODELS_ALL = list(S.drop_duplicates('model').assign(o=lambda d: d.family.map(fam_order)).sort_values(['o', 'model']).model)
ROW_ORDER = ['orijinal'] + sorted(S.loc[S.bg_group == 'astım', 'arka_plan'].unique()) + sorted(S.loc[S.bg_group == 'sağlıklı', 'arka_plan'].unique())
COL = {'orijinal': 'black', 'astım': 'tab:red', 'sağlıklı': 'tab:blue'}

item = W.Dropdown(options=[('hepsi (7 öğenin ortalaması)', 0)] + [(f'{k} · {MV.ITEMS[k]}', k) for k in sorted(S.slot.unique())],
                  value=sorted(S.slot.unique())[0], description='Ses')
mods = W.SelectMultiple(options=MODELS_ALL, value=tuple(MODELS_ALL), rows=min(len(MODELS_ALL), 10), description='Modeller',
                        layout=W.Layout(width='420px'))
metric = W.ToggleButtons(options=[('Kayma (logit)', 'delta_logit'), ('Astım olasılığı', 'prob')], value='delta_logit')
listen = W.Dropdown(options=ROW_ORDER, description='Dinle', layout=W.Layout(width='520px'))
play = W.Button(description='▶ çal')
out, aout = W.Output(), W.Output()

def view_df():
    d = S[S.model.isin(mods.value)]
    if item.value:
        d = d[d.slot == item.value]
    return d.groupby(['arka_plan', 'bg_group', 'model'], as_index=False)[['delta_logit', 'prob']].mean()

def draw(*_):
    with out:
        clear_output(wait=True)
        if not mods.value:
            print('En az bir model seç.'); return
        d, m = view_df(), metric.value
        T = d.pivot_table(index='arka_plan', columns='model', values=m).reindex(index=ROW_ORDER, columns=list(mods.value))
        lim = float(np.nanmax(np.abs(T.values - (0 if m == 'delta_logit' else 0.5)))) or 1.0
        c = 0 if m == 'delta_logit' else 0.5
        display(T.style.format('{:+.2f}' if m == 'delta_logit' else '{:.2f}')
                 .background_gradient(cmap='coolwarm', axis=None, vmin=c - lim, vmax=c + lim))
        fig, ax = plt.subplots(figsize=(10, 0.38 * len(ROW_ORDER) + 1.6))
        mk = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h']
        for j, mod in enumerate(mods.value):
            dm = d[d.model == mod].set_index('arka_plan').reindex(ROW_ORDER)
            yy = np.arange(len(ROW_ORDER)) + (j - (len(mods.value) - 1) / 2) * 0.08
            ax.scatter(dm[m], yy, c=[COL.get(g, 'gray') for g in dm.bg_group], marker=mk[j % len(mk)], s=40,
                       edgecolor='white', linewidth=0.5, label=mod)
        ax.axvline(c, color='gray', lw=1, ls='--')
        ax.set_yticks(range(len(ROW_ORDER))); ax.set_yticklabels(ROW_ORDER, fontsize=8); ax.invert_yaxis()
        ax.set_xlabel('orijinale göre logit kayması (+ = astıma doğru)' if m == 'delta_logit' else 'astım olasılığı')
        ax.set_title(f"{'hepsi (ortalama)' if not item.value else MV.ITEMS[item.value]} — renk: arka planın bağışçısı "
                     "(kırmızı astım, mavi sağlıklı); şekil: model", fontsize=9)
        ax.legend(fontsize=7, loc='center left', bbox_to_anchor=(1.01, 0.5))
        plt.tight_layout(); plt.show()
        g = d[d.bg_group != 'orijinal'].groupby(['model', 'bg_group'])[m].mean().unstack().reindex(list(mods.value))
        print('Betimsel özet (ortalama; istatistiksel test değil):'); display(g.round(2))

def do_play(_):
    with aout:
        clear_output(wait=True)
        if not item.value:
            print('Dinlemek için tek bir öğe seç.'); return
        tag = 'orig' if listen.value == 'orijinal' else listen.value.split(' · ')[0]
        f = OUT / 'mixes_wav' / f'{item.value}_{MV.ITEMS[item.value]}__{tag}.wav'
        print(f.name); display(Audio(filename=str(f)))

for w in (item, mods, metric):
    w.observe(draw, 'value')
play.on_click(do_play)
display(W.VBox([W.HBox([item, metric]), mods, out, W.HBox([listen, play]), aout]))
draw()

## 6. Tüm skorlar (tablo olarak)
Drive'da `experiments/my_voice_v1/scores_*.csv`. Tek tabloda görmek için:

In [ ]:
display(S[['item', 'arka_plan', 'model', 'prob', 'logit', 'delta_logit']].sort_values(['item', 'model', 'arka_plan']).round(3))